##### 建立 Python 3.11 環境（強烈建議指定版本）
conda create -n graphrag python=3.11 -y

##### 啟動虛擬環境
conda activate graphrag

##### 安裝 dateutil
conda install -c conda-forge python-dateutil -y

##### 安裝核心套件
python -m pip install --upgrade pip
python -m pip install graphrag

##### 安裝輔助工具
python -m pip install pandas tiktoken
python -m pip install datasets  

##### 確認 GraphRAG 安裝成功
python -m pip show graphrag

##### 確認 tiktoken 正常
python -c "import tiktoken; print('ok')"

##### 查看 graphrag 指令
graphrag --help

In [1]:
%load_ext dotenv
%dotenv 專案路徑.env

In [2]:
from openai import OpenAI
client = OpenAI(
    base_url="http://127.0.0.1:13305/api/v1",
    api_key="lemonade"
)

def chat(messages, model="qwen3-it-4b-FLM"):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
#確保套件被裝到 Notebook 正在使用的 Python，而不是另一個命令列環境。
import sys
!{sys.executable} -m pip install datasets

In [4]:
import ssl
import certifi

_original_create_default_context = ssl.create_default_context

def create_certifi_context(*args, **kwargs):
    # 強制使用 certifi，避開損壞的 Windows 憑證庫
    kwargs.pop("cafile", None)
    kwargs.pop("capath", None)
    kwargs.pop("cadata", None)

    return _original_create_default_context(
        *args,
        cafile=certifi.where(),
        **kwargs,
    )

ssl.create_default_context = create_certifi_context

In [5]:
from datasets import load_dataset

ds = load_dataset("bigbio/medhop", "medhop_source")

print(ds)
print(ds['train'][0])

# 取得訓練集大小
print(f"Train size: {len(ds['train'])}")

In [6]:
item = ds['train'][0]

print(type(item))
print(item.keys() if hasattr(item, "keys") else item)

print("ID:", item["id"])
print("Question:", item["query"])
print("Answer:", item["answer"])
print("Candidates:", item["candidates"])

print("supports type:", type(item["supports"]))
print("Support count:", len(item["supports"]))
print("First support:", item["supports"][0])

In [7]:
def support_to_text(support):
    if isinstance(support, str):
        return support

    if isinstance(support, (list, tuple)):
        return "\n".join(map(str, support))

    if isinstance(support, dict):
        title = support.get("title", "")
        text = support.get("text", support.get("content", ""))
        return f"{title}\n{text}".strip()

    return str(support)


documents = [
    support_to_text(support)
    for support in item["supports"]
]

context = "\n\n".join(documents)

print(context[:2000])

In [8]:
# medhop_source 欄位：
# id, query, candidates, answer, supports

def medhop_item_to_text(item, include_answer=True):
    candidates_text = "\n".join(
        f"{i + 1}. {candidate}"
        for i, candidate in enumerate(item["candidates"])
    )

    supports_text = "\n\n".join(
        f"[Supporting Document {i + 1}]\n{support}"
        for i, support in enumerate(item["supports"])
    )

    answer_text = (
        f"\n\nGold Answer:\n{item['answer']}"
        if include_answer
        else ""
    )

    return f"""ID:
{item["id"]}

Question:
{item["query"]}

Candidate Answers:
{candidates_text}

Supporting Documents:
{supports_text}{answer_text}
""".strip()

In [ ]:
#預覽前 5,000 個字元，檢查標題、候選答案、支持文件和答案是否排列正確，不會寫入檔案。
item = ds["train"][0]

text = medhop_item_to_text(item)

print(text[:5000])

ID:
MH_train_0

Question:
interacts_with DB00773?

Candidate Answers:
1. DB00072
2. DB00294
3. DB00338
4. DB00341
5. DB00588
6. DB00820
7. DB02546
8. DB02901
9. DB04844

Supporting Documents:
[Supporting Document 1]
Induction of apoptosis of Beta cells of the pancreas by advanced glycation end-products , important mediators of chronic complications of diabetes mellitus . We herein report cytotoxicity of advanced glycation end-products ( AGEs ) on pancreatic beta cells . AGEs stimulated reactive oxygen species ( ROS ) generation but did not arrest proliferation of the P01308 -1 cell line . Pancreatic beta cell lines or primary cultured islets possess a receptor for P51606 ( RAGE ) , and its expression increased after P51606 treatment . TUNEL staining and FACS analysis using annexin V/PI antibodies showed that apoptosis increased in P01308 -1 cells or primary cultured islets when incubated with BSA conjugated with glyceraldehyde ( AGE2 ) or glucoaldehyde ( AGE3 ) , compared with those co

### 資料前處裡

In [10]:
#確認實驗目錄
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\karen\Desktop\Medhop_npu\graphrag_npu_0722"
)

INPUT_DIR = PROJECT_ROOT / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Input:", INPUT_DIR)

Project: C:\Users\karen\Desktop\Medhop_npu\graphrag_npu_0722
Input: C:\Users\karen\Desktop\Medhop_npu\graphrag_npu_0722\input


In [11]:
#資料轉換函式 ，先不要加入 answer，避免答案洩漏。
def medhop_item_to_text(item, include_answer=False):
    candidates_text = "\n".join(
        f"{i + 1}. {candidate}"
        for i, candidate in enumerate(item["candidates"])
    )

    supports_text = "\n\n".join(
        f"[Supporting Document {i + 1}]\n{support}"
        for i, support in enumerate(item["supports"])
    )

    parts = [
        f"ID:\n{item['id']}",
        f"Question:\n{item['query']}",
        f"Candidate Answers:\n{candidates_text}",
        f"Supporting Documents:\n{supports_text}",
    ]

    if include_answer:
        parts.append(f"Gold Answer:\n{item['answer']}")

    return "\n\n".join(parts)

In [ ]:
# 清空input用
# 警告：這會刪除 INPUT_DIR 中所有 doc_*.txt
files_to_remove = sorted(INPUT_DIR.glob("doc_*.txt"))

for file in files_to_remove:
    file.unlink()

print(f"Deleted: {len(files_to_remove)} files")

In [ ]:
# 要輸出的資料筆數
DATA_SIZE = 20

# 確認輸入資料夾目前沒有 doc_*.txt，避免意外覆蓋
existing_files = sorted(INPUT_DIR.glob("doc_*.txt"))

if existing_files:
    raise FileExistsError(
        f"{INPUT_DIR} 已有 {len(existing_files)} 個 doc_*.txt，"
        "請先確認或備份後再執行。"
    )

# 將前 20 筆 MedHop 資料轉成 UTF-8 純文字文件
for index, item in enumerate(ds["train"].select(range(DATA_SIZE))):
    text = medhop_item_to_text(item, include_answer=False)
    output_file = INPUT_DIR / f"doc_{index:04d}.txt"
    output_file.write_text(text, encoding="utf-8")

# 檢查產生結果
generated_files = sorted(INPUT_DIR.glob("doc_*.txt"))

print(f"Generated: {len(generated_files)} files")

for file in generated_files:
    print(f"{file.name}: {file.stat().st_size:,} bytes")

Generated: 20
doc_0000.txt 78898 bytes
doc_0001.txt 32903 bytes
doc_0002.txt 87207 bytes
doc_0003.txt 98229 bytes
doc_0004.txt 55570 bytes
doc_0005.txt 34551 bytes
doc_0006.txt 36417 bytes
doc_0007.txt 79296 bytes
doc_0008.txt 45394 bytes
doc_0009.txt 38465 bytes
doc_0010.txt 45497 bytes
doc_0011.txt 25228 bytes
doc_0012.txt 91362 bytes
doc_0013.txt 29472 bytes
doc_0014.txt 83795 bytes
doc_0015.txt 55484 bytes
doc_0016.txt 65013 bytes
doc_0017.txt 46434 bytes
doc_0018.txt 47521 bytes
doc_0019.txt 96697 bytes


In [13]:
#檢查第一個文件
sample_file = sorted(INPUT_DIR.glob("*.txt"))[0]

print(sample_file)
print(sample_file.read_text(encoding="utf-8")[:3000])

C:\Users\karen\Desktop\Medhop_npu\graphrag_npu_0722\input\doc_0000.txt
ID:
MH_train_0

Question:
interacts_with DB00773?

Candidate Answers:
1. DB00072
2. DB00294
3. DB00338
4. DB00341
5. DB00588
6. DB00820
7. DB02546
8. DB02901
9. DB04844

Supporting Documents:
[Supporting Document 1]
Induction of apoptosis of Beta cells of the pancreas by advanced glycation end-products , important mediators of chronic complications of diabetes mellitus . We herein report cytotoxicity of advanced glycation end-products ( AGEs ) on pancreatic beta cells . AGEs stimulated reactive oxygen species ( ROS ) generation but did not arrest proliferation of the P01308 -1 cell line . Pancreatic beta cell lines or primary cultured islets possess a receptor for P51606 ( RAGE ) , and its expression increased after P51606 treatment . TUNEL staining and FACS analysis using annexin V/PI antibodies showed that apoptosis increased in P01308 -1 cells or primary cultured islets when incubated with BSA conjugated with gly

##### 初始化專案目錄
graphrag init --root C:\Users\karen\Desktop\Medhop_npu\graphrag_npu_0722

##### 查看產生的目錄結構
dir C:\Users\karen\Desktop\Medhop_npu\graphrag_npu_0721

#### 執行 Indexing
graphrag index --root ./graphrag_npu_0722 --verbose